In [1]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 

In [ ]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, context_size, use_embedding=False, embedding_dim=10):
        super().__init__()
        self.use_embedding = use_embedding

        if self.use_embedding:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
            self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        else:
            self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True, nonlinearity='tanh')
        
        self.linear = nn.Linear(hidden_size, vocab_size)

        ### use orthogonal initialization of weights ### 
        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h=None):
        if self.use_embedding:
            embedded = self.embedding(x)
            out, h = self.rnn(embedded, h)
        else:
            out, h = self.rnn(x, h)

        out = self.linear(out[:,-1,:])

        return out, h  
    
class RNNDecoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, use_embedding=False, embedding_dim=10):
        super().__init__()
        self.use_embedding = use_embedding

        if self.use_embedding:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
            self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='relu')
        else:
            self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True, nonlinearity='relu')

        self.fc = nn.Linear(hidden_size, vocab_size)

        ### use orthogonal initialization of weights ###
        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h):
        if self.use_embedding:
            embedded = self.embedding(x)
            out, _ = self.rnn(embedded, h)
        else:
            out, _ = self.rnn(x, h)

        return self.fc(out) 
    
class RNNAutoencoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, use_embedding=False, embedding_dim=10 ):
        super().__init__()
        self.encoder = RNNEncoder(vocab_size, embedding_dim, hidden_size, use_embedding, embedding_dim)
        self.decoder = RNNDecoder(vocab_size, embedding_dim, hidden_size, use_embedding, embedding_dim)
    
    def forward(self, x, decoder_input, h=None):
        next_token, h = self.encoder(x, h)
        decoder_output = self.decoder(decoder_input, h)
        return next_token, decoder_output, h

In [ ]:
total_layers = 3
model = {}